# 중급 프로젝트 데이터 처리

## 📋 목차
- 0. 라이브러리 설치 및 경로 설정
- 1. 텍스트 클리닝
- 2. HWP 파싱 및 문서 구조화
- 3. PDF 파싱 및 청킹
- 4. HWP 청킹
- 5. 통합 및 저장


---
## 0. 라이브러리 설치 및 경로 설정

### ⚠️ 처음 실행시 데이터 저장

아래 순서대로 진행해주세요.

1. JupyterLab 왼쪽 파일 패널에서 `AI-based-RFP-RAG-System` 폴더 들어간 후
   상단 폴더 아이콘 클릭해서 `data` 폴더 생성

2. 구글 드라이브 공유 문서함 > 데이터셋 > 중급에서 아래 파일 다운로드
   - `original_data_list.zip`
   - `data_list_advanced.xlsx`

3. JupyterLab에서 만든 `data` 폴더에 위의 파일 두개를 업로드

이후 노트북 순서대로 실행하면 됩니다.


In [ ]:
import chromadb, json, numpy as np, os
from tqdm import tqdm

CHROMA_PATH     = "/Users/who/Desktop/code_it/project01/final_files/chroma_seol_qwen3"
COLLECTION_NAME = "rfp_seol_chunks_qwen3"
EXPORT_DIR      = "/Users/who/Desktop/code_it/project01/final_files/data"

# 데이터 로드
embeddings = np.load(os.path.join(EXPORT_DIR, "embeddings.npy")).tolist()
with open(os.path.join(EXPORT_DIR, "chroma_export.json"), "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"로드 완료: {len(data['ids']):,}개")

# ChromaDB 신규 생성
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"}
)
print(f"컬렉션 생성 완료")

# 벡터 삽입
BATCH = 500
for i in tqdm(range(0, len(data["ids"]), BATCH), desc="삽입 중"):
    collection.add(
        ids        = data["ids"]       [i:i+BATCH],
        documents  = data["documents"] [i:i+BATCH],
        metadatas  = data["metadatas"] [i:i+BATCH],
        embeddings = embeddings        [i:i+BATCH],
    )

print(f"\n✅ 완료 — {collection.count():,}개 저장")

로드 완료: 51,366개
컬렉션 생성 완료


삽입 중: 100%|██████████| 103/103 [00:54<00:00,  1.89it/s]


✅ 완료 — 51,366개 저장


In [4]:
# ChromaDB 저장 검증
print(f"✅ 총 저장 청크: {collection.count():,}개")

# 샘플 5개 확인
sample = collection.get(limit=5, include=["documents", "metadatas"])
print(f"\n📋 샘플 청크 확인 (5개)")
for i, (doc, meta) in enumerate(zip(sample["documents"], sample["metadatas"])):
    print(f"\n  [{i+1}] {meta.get('발주기관', '')} | {meta.get('사업명', '')[:30]}")
    print(f"       chunk_type: {meta.get('chunk_type', '')}")
    print(f"       내용: {doc[:60].strip()}")

# chunk_type 분포 (배치 처리)
type_dist = {}
BATCH     = 1000
total     = collection.count()

for offset in range(0, total, BATCH):
    batch = collection.get(
        limit   = BATCH,
        offset  = offset,
        include = ["metadatas"]
    )
    for m in batch["metadatas"]:
        t = m.get("chunk_type", "unknown")
        type_dist[t] = type_dist.get(t, 0) + 1

print(f"\n📊 chunk_type 분포")
for k, v in sorted(type_dist.items()):
    print(f"   {k}: {v:,}개")

print("\n✅ 검증 완료")

✅ 총 저장 청크: 51,366개

📋 샘플 청크 확인 (5개)

  [1] 한영대학 | 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학
       chunk_type: summary
       내용: [사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 | 발주기관: 한영대학]

  [2] 한영대학 | 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학
       chunk_type: summary
       내용: | 2 추진배경 및 필요성 |
| --- |
| 2 추진배경 및 필요성 |
| --- |

학사제도･제도개편

  [3] 한영대학 | 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학
       chunk_type: normal
       내용: | 3 구축범위 |
| --- |
추진일정은 발주기관 와 계약상대자 의 상호협의를 통해 변경할 수 있음.


  [4] 한영대학 | 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학
       chunk_type: normal
       내용: | 1 제안 요청 개요 |
| --- |
| Ⅲ 제안 요청 내용 |
| --- |

     

| 1 제안

  [5] 한영대학 | 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학
       chunk_type: summary
       내용: | 1 입찰 및 계약방법 |
| --- |
□ 입찰방식
1) 사업자 선정 방식
○ 기본방침
- 최저입찰제에서

📊 chunk_type 분포
   normal: 35,429개
   summary: 12,439개
   table: 3,498개

✅ 검증 완료


In [5]:
import chromadb
print(chromadb.__version__)

1.5.9


In [4]:
import chromadb, json, numpy as np, os
from tqdm import tqdm

CHROMA_PATH     = "/Users/who/Desktop/code_it/project01/final_files/chroma_seol_qwen3"
COLLECTION_NAME = "rfp_seol_chunks_qwen3"
EXPORT_DIR      = "/Users/who/Desktop/code_it/project01/final_files/data"

# 데이터 로드
embeddings = np.load(os.path.join(EXPORT_DIR, "embeddings.npy")).tolist()
with open(os.path.join(EXPORT_DIR, "chroma_export.json"), "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"로드 완료: {len(data['ids']):,}개")

# 기존 폴더 초기화
import shutil
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
os.makedirs(CHROMA_PATH)

# ChromaDB 구축
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"}
)

BATCH = 500
for i in tqdm(range(0, len(data["ids"]), BATCH), desc="삽입 중"):
    collection.add(
        ids        = data["ids"]       [i:i+BATCH],
        documents  = data["documents"] [i:i+BATCH],
        metadatas  = data["metadatas"] [i:i+BATCH],
        embeddings = embeddings        [i:i+BATCH],
    )

print(f"\n✅ 완료 — {collection.count():,}개 저장")

로드 완료: 51,366개


InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [11]:
import sys
!{sys.executable} -m pip install olefile openpyxl pymupdf pdfplumber tqdm langchain-text-splitters tiktoken -q


In [12]:
import os
import re
import pandas as pd

BASE_DIR = "/Users/who/Desktop/code_it/project01/final_files"
DATA_DIR      = os.path.join(BASE_DIR, "data", "files_advanced")
METADATA_PATH = os.path.join(BASE_DIR, "data", "data_list_advanced.xlsx")

print("DATA_DIR 존재 :", os.path.exists(DATA_DIR))
print("METADATA 존재 :", os.path.exists(METADATA_PATH))


DATA_DIR 존재 : True
METADATA 존재 : True


In [13]:
import zipfile

zip_path     = os.path.expanduser("~/AI-based-RFP-RAG-System/data/original_data_list.zip")
extract_path = os.path.expanduser("~/AI-based-RFP-RAG-System/data")

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ 압축 해제 완료")
else:
    print("⚠️ zip 파일 없음 — 이미 압축 해제되어 있거나 파일 경로 확인 필요")


⚠️ zip 파일 없음 — 이미 압축 해제되어 있거나 파일 경로 확인 필요


In [14]:
df = pd.read_excel(METADATA_PATH)
hwp_df = df[df["파일형식"] == "hwp"].reset_index(drop=True)
pdf_df = df[df["파일형식"] == "pdf"].reset_index(drop=True)

print(f"전체: {len(df)}개 (HWP: {len(hwp_df)}, PDF: {len(pdf_df)})")


전체: 690개 (HWP: 665, PDF: 25)


---
## 1. 텍스트 클리닝


In [15]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 0. HWP 제어문자 제거 (\n 제외)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', text)
    # 0. PDF 소프트 하이픈 제거
    text = text.replace('\xad', '')
    # 1. 한자 + 깨진 문자 제거
    text = re.sub(r'[\u4E00-\u9FFF\u3400-\u4DBF]\u0203', ' ', text)
    text = re.sub(r'\u0203', ' ', text)
    text = re.sub(r'[\u4E00-\u9FFF\u3400-\u4DBF]', ' ', text)
    # 2. 라틴 확장 문자 제거
    text = re.sub(r'[\u0100-\u024F]', ' ', text)
    # 3. 그리스 문자 제거
    text = re.sub(r'[\u0370-\u03FF\u1F00-\u1FFF]', ' ', text)
    # 4. 기타 특수기호 제거
    text = re.sub(r'[\u2000-\u206F\u2100-\u214F]', ' ', text)
    # 5. 목차 점선 및 페이지 번호 제거
    text = re.sub(r'[·.]{3,}\s*\d*', ' ', text)
    text = re.sub(r'-\s*\d+\s*-', ' ', text)
    text = re.sub(r'\d+\s*페이지', ' ', text)
    # 6. 연속 줄바꿈 → 최대 2개로 압축
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(\n\s*){3,}', '\n\n', text)  # 추가
    # 7. 연속 공백 → 단일 공백
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()
print("✅ 클리닝 함수 정의 완료")


✅ 클리닝 함수 정의 완료


In [16]:
df["텍스트_클리닝"] = df["텍스트"].apply(clean_text)

df["글자수_원본"]   = df["텍스트"].astype(str).apply(len)
df["글자수_클리닝"] = df["텍스트_클리닝"].apply(len)

print("=== 클리닝 전후 글자 수 비교 ===")
print(f"원본 평균   : {df['글자수_원본'].mean():,.0f}자")
print(f"클리닝 후   : {df['글자수_클리닝'].mean():,.0f}자")
print(f"평균 감소량 : {(df['글자수_원본'] - df['글자수_클리닝']).mean():,.0f}자")
print()
print("✅ 전체 클리닝 완료")


=== 클리닝 전후 글자 수 비교 ===
원본 평균   : 4,188자
클리닝 후   : 3,933자
평균 감소량 : 255자

✅ 전체 클리닝 완료


In [17]:
encoding_errors = 0
for text in df["텍스트_클리닝"]:
    try:
        text.encode("utf-8")
    except Exception:
        encoding_errors += 1

print(f"UTF-8 인코딩 오류: {encoding_errors}개")
print("✅ 인코딩 확인 완료")


UTF-8 인코딩 오류: 0개
✅ 인코딩 확인 완료


---
## 2. HWP 파싱 및 문서 구조화


In [43]:
# Cell 12 — HWP 파서 (hwp5txt primary + 바이너리 표 추출 보완)
import olefile
import zlib
import struct
import subprocess
import re
from tqdm import tqdm

# ── HWP 레코드 TagID 상수 ─────────────────────────────────────────
TAG_PARA_TEXT   = 67
TAG_CTRL_HEADER = 68
TAG_TABLE       = 69
TAG_CELL        = 71

# ── 레코드 파싱 ───────────────────────────────────────────────────
def iter_records(data: bytes):
    """HWP BodyText 스트림을 레코드 단위로 순회."""
    pos = 0
    while pos + 4 <= len(data):
        header = struct.unpack_from("<I", data, pos)[0]
        tag_id = header & 0x3FF
        level  = (header >> 10) & 0x3FF
        size   = (header >> 20) & 0xFFF
        pos += 4
        if size == 0xFFF:
            if pos + 4 > len(data):
                break
            size = struct.unpack_from("<I", data, pos)[0]
            pos += 4
        payload = data[pos: pos + size]
        pos += size
        yield tag_id, level, size, payload

# ── 텍스트 디코딩 ─────────────────────────────────────────────────
def decode_text(payload: bytes) -> str:
    """TAG_PARA_TEXT 페이로드 → 문자열 (UTF-16LE)"""
    try:
        return payload.decode("utf-16-le")
    except Exception:
        return ""

# ── 섹션 스트림 압축 해제 ─────────────────────────────────────────
def decompress_section(ole, section_name: str) -> bytes:
    try:
        raw    = ole.openstream(section_name).read()
        header = ole.openstream("FileHeader").read()
        flags  = struct.unpack_from("<I", header, 36)[0]
        compressed = bool(flags & 0x1)
        if compressed:
            return zlib.decompress(raw, -15)
        else:
            return raw
    except Exception:
        return b""

# ── 표 추출 ───────────────────────────────────────────────────────
def extract_table(records_iter, table_level: int):
    rows               = []
    current_row        = []
    current_cell_texts = []
    prev_x             = None

    for tag_id, level, size, payload in records_iter:
        if tag_id == 71 and level == table_level:
            if len(payload) >= 6:
                x = struct.unpack_from("<H", payload, 4)[0]
            else:
                x = (prev_x or 0) + 1

            if current_cell_texts:
                current_row.append(" ".join(current_cell_texts).strip())
                current_cell_texts = []
            elif prev_x is not None:
                current_row.append("")

            if x == 0 and prev_x is not None:
                if current_row:
                    rows.append(current_row)
                    current_row = []

            prev_x = x

        elif tag_id == TAG_PARA_TEXT:
            text = decode_text(payload).strip()
            if text:
                current_cell_texts.append(text)

        elif level < table_level:
            if current_cell_texts:
                current_row.append(" ".join(current_cell_texts).strip())
            if current_row:
                rows.append(current_row)
            break

    return rows

# ── 표 → Markdown ─────────────────────────────────────────────────
def table_to_markdown(rows: list) -> str:
    if not rows:
        return ""
    max_cols = max(len(r) for r in rows)
    norm     = [r + [""] * (max_cols - len(r)) for r in rows]
    lines    = []
    lines.append("| " + " | ".join(norm[0]) + " |")
    lines.append("| " + " | ".join(["---"] * max_cols) + " |")
    for row in norm[1:]:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)

# ── 섹션 파싱 ─────────────────────────────────────────────────────
def parse_section(data: bytes) -> str:
    parts   = []
    records = iter_records(data)

    for tag_id, level, size, payload in records:
        if tag_id == TAG_PARA_TEXT:
            text = decode_text(payload).strip()
            if text:
                parts.append(text)
        elif tag_id == 69 and size % 36 == 0:
            table_rows = extract_table(records, table_level=level)
            if table_rows:
                parts.append("\n" + table_to_markdown(table_rows) + "\n")

    return "\n".join(parts)

# ── 표 유효성 확인 ────────────────────────────────────────────────
def has_table_content(table_md: str) -> bool:
    """표에 실제 한글/영문/숫자 내용이 있는지 확인"""
    for line in table_md.split("\n"):
        if not line.startswith("|") or "---" in line:
            continue
        for cell in line.split("|"):
            if re.search(r'[가-힣a-zA-Z0-9]', cell):
                return True
    return False

# ── 표만 추출 (바이너리 파서) ─────────────────────────────────────
def parse_hwp_tables_only(filepath: str) -> str:
    """바이너리 파서로 표만 추출 → Markdown 형식"""
    try:
        ole = olefile.OleFileIO(filepath)
    except Exception:
        return ""

    all_tables  = []
    section_idx = 0
    while True:
        stream_name = f"BodyText/Section{section_idx}"
        if not ole.exists(stream_name):
            break
        data = decompress_section(ole, stream_name)
        if data:
            records = iter_records(data)
            for tag_id, level, size, payload in records:
                if tag_id == 69 and size % 36 == 0:
                    table_rows = extract_table(records, table_level=level)
                    if table_rows:
                        md      = table_to_markdown(table_rows)
                        cleaned = re.sub(r'[\u4E00-\u9FFF\u3400-\u4DBF]', ' ', md)
                        cleaned = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', cleaned)
                        cleaned = re.sub(r' {2,}', ' ', cleaned)
                        if has_table_content(cleaned):
                            all_tables.append(cleaned)
        section_idx += 1

    ole.close()
    return "\n\n".join(all_tables)

# ── HWP 파일 전체 파싱 ───────────────────────────────────────────
def parse_hwp(filepath: str) -> str:
    """
    HWP 파일 → 전체 텍스트 반환
    - 본문 텍스트: hwp5txt (clean Korean text)
    - 표:         바이너리 파서 (Markdown 변환)
    """
    # 1. hwp5txt로 본문 텍스트 추출
    try:
        result     = subprocess.run(
            ["hwp5txt", str(filepath)],
            capture_output=True, text=True, timeout=60
        )
        main_text  = result.stdout.strip()
        main_text  = re.sub(r'<표>', '', main_text)
        main_text  = re.sub(r'<그림>', '', main_text)
        main_text  = re.sub(r'\n{3,}', '\n\n', main_text).strip()
    except Exception:
        main_text = ""

    # 2. 바이너리 파서로 표 추출
    tables_text = parse_hwp_tables_only(filepath)

    # 3. 결합
    if tables_text:
        return main_text + "\n\n" + tables_text
    return main_text

print("✅ Cell 12 완료 — HWP 파서 재정의 (hwp5txt primary + 바이너리 표 추출)")

✅ Cell 12 완료 — HWP 파서 재정의 (hwp5txt primary + 바이너리 표 추출)


In [44]:
# ── 전체 HWP 파일 배치 파싱 ─────────────────────────────────────
hwp_parsed   = {}
parse_errors = []
hwp_files    = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".hwp")]

for fname in tqdm(hwp_files, desc="HWP 파싱"):
    fpath = os.path.join(DATA_DIR, fname)
    try:
        hwp_parsed[fname] = parse_hwp(fpath)
    except Exception as e:
        parse_errors.append((fname, str(e)))
        hwp_parsed[fname] = ""

success     = sum(1 for t in hwp_parsed.values() if len(t) > 100)
table_files = sum(1 for t in hwp_parsed.values() if "| --- |" in t)
empty       = sum(1 for t in hwp_parsed.values() if len(t) <= 100)

print(f"\n✅ 파싱 완료")
print(f"  전체 파일   : {len(hwp_files)}개")
print(f"  텍스트 추출 : {success}개")
print(f"  표 포함 파일: {table_files}개  ← advanced 대비 개선 핵심")
print(f"  텍스트 없음 : {empty}개")
if parse_errors:
    print(f"  파싱 오류   : {len(parse_errors)}개")
    for fname, err in parse_errors[:5]:
        print(f"    - {fname}: {err}")

HWP 파싱:   0%|          | 0/665 [00:00<?, ?it/s]

HWP 파싱: 100%|██████████| 665/665 [16:46<00:00,  1.51s/it]


✅ 파싱 완료
  전체 파일   : 665개
  텍스트 추출 : 665개
  표 포함 파일: 664개  ← advanced 대비 개선 핵심
  텍스트 없음 : 0개


In [45]:
# remove_empty_tables — 한글/영문/숫자 기준으로 표 유효성 판단 (수정)
def remove_empty_tables(text: str) -> str:
    lines  = text.split("\n")
    result = []
    i = 0
    while i < len(lines):
        if lines[i].startswith("|"):
            table_lines = []
            while i < len(lines) and lines[i].startswith("|"):
                table_lines.append(lines[i])
                i += 1
            # ★ 수정: 구분선 행 제외하고 한글/영문/숫자 내용 확인
            has_content = False
            for line in table_lines:
                if "---" in line:
                    continue
                for cell in line.split("|"):
                    if re.search(r'[가-힣a-zA-Z0-9]', cell):
                        has_content = True
                        break
                if has_content:
                    break
            if has_content:
                result.extend(table_lines)
        else:
            result.append(lines[i])
            i += 1
    return "\n".join(result)

In [46]:
# ── 원본 파싱 텍스트 기반 문서 구조화 ───────────────────────────
# 기존 clean_text 에서 Markdown 표 행(| 로 시작)은 클리닝 스킵 추가
    
# clean_text_hwp — 제어문자 및 null byte 제거 추가 (수정)
def clean_text_hwp(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # ★ 추가: null byte 및 제어문자 제거 (hwp5txt 출력에도 잔존할 수 있음)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', text)

    lines   = text.split("\n")
    cleaned = []
    for line in lines:
        if line.startswith("|"):
            line = re.sub(r'[\u4E00-\u9FFF\u3400-\u4DBF]', ' ', line)
            line = re.sub(r'[\u0100-\u024F]', ' ', line)
            line = re.sub(r'[\u2000-\u206F\u2100-\u214F]', ' ', line)
            line = re.sub(r' {2,}', ' ', line)
            cleaned.append(line)
            continue

        line = re.sub(r'[\u4E00-\u9FFF\u3400-\u4DBF]', ' ', line)
        line = re.sub(r'[\u0100-\u024F]', ' ', line)
        line = re.sub(r'[\u2000-\u206F\u2100-\u214F]', ' ', line)
        line = re.sub(r'[\u4E00-\u9FFF][\u3400-\u4DBF]?[a-zA-Z]', ' ', line)
        line = re.sub(r' {2,}', ' ', line)
        cleaned.append(line)

    text = "\n".join(cleaned)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = remove_empty_tables(text)
    text = re.sub(r'\n\s+\n', '\n\n', text)
    return text.strip()


def build_document(row, doc_id: int) -> dict:
    사업명   = str(row.get("사업명", "")).strip()
    발주기관 = str(row.get("발주 기관", "")).strip()
    사업요약 = str(row.get("사업 요약", "")).strip()
    사업금액 = str(row.get("사업 금액", "")).strip()
    공고번호 = str(row.get("공고 번호", "")).strip()
    파일형식 = str(row.get("파일형식", "")).strip()
    파일명   = str(row.get("파일명", "")).strip()

    # 원본 파싱 텍스트 우선, 없으면 advanced 텍스트로 폴백
    raw_text = hwp_parsed.get(파일명, "")
    if len(raw_text) < 200:
        raw_text = str(row.get("텍스트_클리닝", ""))

    텍스트 = clean_text_hwp(raw_text)

    has_full_text = len(텍스트) > 500
    main_text     = 텍스트 if has_full_text else 사업요약

    return {
        "doc_id"       : doc_id,
        "사업명"       : 사업명,
        "발주기관"     : 발주기관,
        "사업금액"     : 사업금액,
        "공고번호"     : 공고번호,
        "파일형식"     : 파일형식,
        "사업요약"     : 사업요약,
        "text"         : main_text,
        "has_full_text": has_full_text,
        "has_table"    : "| --- |" in 텍스트,
    }


hwp_df_clean = df[df["파일형식"] == "hwp"].reset_index(drop=True)
documents = [build_document(row, i) for i, row in hwp_df_clean.iterrows()]

table_docs = sum(1 for d in documents if d["has_table"])
print(f"✅ 문서 구조화 완료: {len(documents)}건")
print(f"  - 본문 사용    : {sum(1 for d in documents if d['has_full_text'])}건")
print(f"  - 요약 사용    : {sum(1 for d in documents if not d['has_full_text'])}건")
print(f"  - 표 포함 문서 : {table_docs}건")

✅ 문서 구조화 완료: 665건
  - 본문 사용    : 625건
  - 요약 사용    : 40건
  - 표 포함 문서 : 0건


In [47]:
patterns = {
    # 금액 — 만원/억 단위는 최소 2자리, 원 단위는 최소 5자리(10,000원 이상)
    "금액": r'(?:[1-9][\d,]+원|[1-9]\d*(?:억|만원|백만원|천만원))',
    "날짜": r'\d{4}년\s*\d{1,2}월\s*\d{1,2}일?',
    # 기간 — 최대 60개월, 52주 이내
    "기간": r'(?:[1-9]|[1-5]\d)\s*(?:개월|주)',
    "비율": r'\d+\.?\d*\s*%',
    # 수량 — 앞에 0으로 시작하는 숫자 제외
    "수량": r'[1-9][\d,]*\s*(?:개|명|건|식|대|세트)',
}

def find_numbers_hwp(text):
    normalized = re.sub(r'\s+', ' ', text)
    found = {}
    for name, pattern in patterns.items():
        matches = re.findall(pattern, normalized)
        if matches:
            found[name] = matches
    return found

def make_numeric_summary(found: dict) -> str:
    lines = []
    for key in ["금액", "기간", "수량", "비율", "날짜"]:
        if key in found:
            values = [v.strip() for v in found[key]
                     if v.strip() and re.search(r'\d', v)]
            if values:
                lines.append(f"{key}: {', '.join(set(values))}")
    return " | ".join(lines)

print("✅ HWP 수치 패턴 함수 정의 완료")


✅ HWP 수치 패턴 함수 정의 완료


In [48]:
# 수치 패턴 감지 (청킹 전 문서 단위로 실행 — 청킹 후 청크에도 적용)
# 수치 패턴 함수 정의 + 동작 확인용 

number_heavy_docs = []
for doc in documents:
    found = find_numbers_hwp(doc["text"])
    if found:
        doc["numeric_summary"] = make_numeric_summary(found)
        number_heavy_docs.append((doc, found))

print(f"수치 패턴 있는 HWP 문서: {len(number_heavy_docs)}개")
for doc, found in number_heavy_docs[:3]:
    print(f"사업명: {doc['사업명']}")
    print(f"numeric_summary: {doc['numeric_summary']}")
    print()


수치 패턴 있는 HWP 문서: 527개
사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화
numeric_summary: 금액: 130,000,000원 | 기간: 1개월, 3개월 | 수량: 1개, 3개

사업명: 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선
numeric_summary: 금액: 129,300,000원, 37,000,000원, 92,300,000원

사업명: 도시계획위원회 통합관리시스템 구축용역
numeric_summary: 금액: 150,000,000원 | 수량: 2개



In [49]:
def inject_title(doc: dict) -> dict:
    """수치 처리 완료 후 텍스트 앞에 [사업명 | 발주기관] 삽입"""
    prefix = f"[사업명: {doc['사업명']} | 발주기관: {doc['발주기관']}]\n\n"
    doc["text"] = prefix + doc["text"]
    return doc

documents = [inject_title(doc) for doc in documents]
print("✅ 제목/섹션 삽입 완료")
print()
print("=== 샘플 확인 ===")
print(documents[0]["text"][:300])


✅ 제목/섹션 삽입 완료

=== 샘플 확인 ===
[사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 | 발주기관: 한영대학]

2024년 특성화 맞춤형 교육환경 구축 트랙운영 학사정보시스템 고도화
 제안요청서

2024. 10.

목 차
Ⅰ. 사업 안내 - 1
 1. 사업개요 - 1
 2. 추진배경 및 필요성 - 1
 3. 기대효과 - 1
 Ⅱ. 구축 방안 - 2
 1. 구축목표 - 2
 2. 구축일정 - 2
 3. 구축범위 - 3
 Ⅲ. 제안 요청 내용 - 4
 1. 제안 요청 개요 - 4
 Ⅳ. 제안안내 사항 - 5
 1. 입찰 및 계약방법 -


---
## 3. PDF 파싱 및 청킹


In [50]:
import pdfplumber

# 경로 설정 (공통 환경변수 재사용)
pdf_df = df[df["파일형식"] == "pdf"].reset_index(drop=True)
print(f"  PDF 파일 수: {len(pdf_df)}개")

  PDF 파일 수: 25개


In [51]:
def table_to_markdown(table):
    """2차원 리스트(표)를 Markdown 문자열로 변환"""
    if not table or len(table) < 2:
        return ""
    headers = [str(h) if h else "" for h in table[0]]
    rows    = table[1:]
    md  = "| " + " | ".join(headers) + " |\n"
    md += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    for row in rows:
        md += "| " + " | ".join(str(c) if c else "" for c in row) + " |\n"
    return md


def extract_tables_from_pdf(pdf_path):
    """PDF 원본에서 표 감지 및 Markdown 변환, 페이지별 텍스트도 함께 반환"""
    table_results = []
    text_by_page  = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # 표 감지 및 Markdown 변환
                tables = page.extract_tables()
                for table in tables:
                    if not table or len(table) < 2:
                        continue
                    md = table_to_markdown(table)
                    if md.strip():
                        table_results.append({
                            "page"    : page_num + 1,
                            "markdown": md
                        })
                # 페이지 텍스트 추출
                text = page.extract_text() or ""
                text_by_page.append({
                    "page": page_num + 1,
                    "text": text
                })
    except Exception as e:
        print(f"  오류: {e}")
    return table_results, text_by_page

print("  표 감지 및 변환 함수 정의 완료")

  표 감지 및 변환 함수 정의 완료


In [52]:
all_table_chunks = []  # 표 청크 저장 리스트
all_text_by_page = {}  # 파일별 페이지 텍스트 저장

for _, row in tqdm(pdf_df.iterrows(), total=len(pdf_df), desc="표 추출 중"):
    pdf_path = os.path.join(DATA_DIR, row["파일명"])
    if not os.path.exists(pdf_path):
        continue

    tables, texts = extract_tables_from_pdf(pdf_path)

    # 표 청크 저장 — HWP 스키마에 맞춰 저장
    doc_id = row.name + 10000  # HWP doc_id와 겹치지 않도록 offset
    for idx, t in enumerate(tables):
        # 수치 패턴 감지 (표 추출 시점에 미리 적용)
        all_table_chunks.append({
            "chunk_id"       : f"pdf_{doc_id}_{idx}",
            "doc_id"         : doc_id,
            "chunk_index"    : idx,
            "text"           : t["markdown"],
            "numeric_summary": "",   # 수치 패턴 감지 후 업데이트
            "chunk_type"     : "table",
            "metadata": {
                "doc_id"      : str(doc_id),
                "사업명"      : str(row.get("사업명", "")),
                "발주기관"    : str(row.get("발주 기관", "")),
                "사업금액"    : str(row.get("사업 금액", "")),
                "공고번호"    : str(row.get("공고 번호", "")),
                "파일형식"    : "pdf",
                "파일명"      : str(row.get("파일명", "")),
                "page"        : str(t["page"]),
                "has_full_text": True,
                "has_table"   : True,
                "chunk_id"    : f"pdf_{doc_id}_{idx}",
                "chunk_index" : str(idx),
            }
        })

    # 페이지별 텍스트 저장
    all_text_by_page[row["파일명"]] = texts

print(f"  표 추출 완료")
print(f"  총 표 청크 수: {len(all_table_chunks)}개")

표 추출 중: 100%|██████████| 25/25 [02:42<00:00,  6.51s/it]

  표 추출 완료
  총 표 청크 수: 3664개


In [53]:
import re

# 수치 패턴 정의 (장표 번호 및 조항 제외 로직 고도화)
number_patterns = {
    "금액": r'\d[\d,]*\s*(?:억|만원|원|백만원|천만원)',
    "기간": r'\d+\s*(?:년|개월|주|일|시간)',
    # 앞에 '제'가 붙거나 뒤에 '패/페/항/조/절'이 붙는 패턴은 수량에서 제외
    "수량": r'(?<!제)\b\d[\d,]*\s*(?:명|개|건|대|식|실|종|권|회)\b(?!패|페|항|조|절)',
    "비율": r'\d+(?:\.\d+)?\s*%',
    "날짜": r'\d{4}[-./년]\s*\d{1,2}[-./월]'
}

def find_numbers(text):
    """텍스트에서 수치 패턴 감지"""
    found = {}
    for key, pattern in number_patterns.items():
        matches = re.findall(pattern, str(text))
        if matches:
            found[key] = matches
    return found

def make_numeric_summary(found: dict) -> str:
    """감지된 수치 패턴을 요약 문자열로 변환"""
    lines = []
    for key in ["금액", "기간", "수량", "비율", "날짜"]:
        if key in found:
            values = [v.strip() for v in found[key] if re.search(r'\d', v)]
            if values:
                lines.append(f"{key}: {', '.join(set(values))}")
    return " | ".join(lines)  # 문법 오류 수정 완료

# 표 청크에 고도화된 수치 패턴 적용
for chunk in all_table_chunks:
    found = find_numbers(chunk["text"])
    chunk["numeric_summary"] = make_numeric_summary(found) if found else ""
    chunk["chunk_type"]      = "summary" if found else "table"

summary_count = len([c for c in all_table_chunks if c["chunk_type"] == "summary"])
print(f"  수치 패턴 감지 완료")
print(f"  수치 있는 표 청크: {summary_count}개")
print(f"  수치 없는 표 청크: {len(all_table_chunks) - summary_count}개")

# 표 청크 numeric_summary 업데이트
for chunk in all_table_chunks:
    found = find_numbers(chunk["text"])
    chunk["numeric_summary"] = make_numeric_summary(found) if found else ""
    if found:
        chunk["chunk_type"] = "summary"

summary_count = sum(1 for c in all_table_chunks if c["chunk_type"] == "summary")
print(f"  수치 있는 표 청크: {summary_count}개")
print(f"  수치 없는 표 청크: {len(all_table_chunks) - summary_count}개")

  수치 패턴 감지 완료
  수치 있는 표 청크: 705개
  수치 없는 표 청크: 2959개
  수치 있는 표 청크: 705개
  수치 없는 표 청크: 2959개


In [54]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE    = 1024
CHUNK_OVERLAP = 200
SEPARATORS    = ["\n\n", "\n", "다. ", "함. ", "임. ", "습니다. ", ". ", " ", ""]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS
)

def extract_section_name(text: str) -> str:
    lines = text.strip().split("\n")
    for line in lines[:5]:
        line = line.strip()
        # 날짜 패턴이면 섹션명으로 보지 않음
        if re.match(r'^\d{4}\.\s*\d{1,2}\.\s*\d{1,2}', line):
            continue
        if re.match(r'^[ⅠⅡⅢⅣⅤ□○\-]?\s*제?\s*\d+\s*[장절조항]?\s*[\.\.\s]\s*\S', line):
            return line[:50]
        if re.match(r'^[ⅠⅡⅢⅣⅤ][\.\s]', line):
            return line[:50]
    return ""
    
def is_noise_chunk(chunk):
    """노이즈 청크 필터"""
    if re.fullmatch(r'[\s\-\d]+', chunk):
        return True
    if chunk.count('  ') > len(chunk) * 0.1:
        return True
    if chunk.count('·') > 5 or chunk.count('..') > 5:
        return True
    return False

# --- 통계 수집을 위한 변수 초기화 ---
pdf_text_chunks = []
noise_stats = {
    "total_splits": 0,
    "filtered_short": 0,
    "filtered_noise": 0
}
# ---------------------------------

for _, row in tqdm(pdf_df.iterrows(), total=len(pdf_df), desc="텍스트 청킹 중"):
    filename = row["파일명"]
    if filename not in all_text_by_page:
        continue

    idx = 0
    for page_info in all_text_by_page[filename]:
        page_num = page_info["page"]
        splits = splitter.split_text(page_info["text"])
        noise_stats["total_splits"] += len(splits)

        for chunk in splits:
            if len(chunk) < 50:
                noise_stats["filtered_short"] += 1
                continue
            if is_noise_chunk(chunk):
                noise_stats["filtered_noise"] += 1
                continue

            found   = find_numbers(chunk)
            numeric = make_numeric_summary(found) if found else ""
            pdf_text_chunks.append({
                "chunk_id"       : f"pdf_text_{row.name + 10000}_{idx}",
                "doc_id"         : str(row.name + 10000),
                "chunk_index"    : idx,
                "text"           : chunk,
                "numeric_summary": numeric,
                "chunk_type"     : "summary" if found else "normal",
                "metadata": {
                    "doc_id"       : str(row.name + 10000),
                    "사업명"       : row["사업명"],
                    "발주기관"     : row["발주 기관"],
                    "파일형식"     : "pdf",
                    "파일명"       : filename,
                    "has_full_text": True,
                    "has_table"    : False,
                    "chunk_id"     : f"pdf_text_{row.name + 10000}_{idx}",
                    "chunk_index"  : str(idx),
                    "section"      : extract_section_name(chunk),
                    "page"         : str(page_num),
                }
            })
            idx += 1

# --- 보고서에 바로 활용할 수 있는 정량적 통계 출력 ---
print(f"\n  텍스트 청킹 및 정제 완료")
print(f"  ==========================================")
print(f"  [전처리 통계] 최초 분할된 총 청크 수 : {noise_stats['total_splits']} 개")
print(f"  [전처리 통계] 길이 미달로 제거된 청크: {noise_stats['filtered_short']} 개")
print(f"  [전처리 통계] 노이즈 패턴으로 제거됨  : {noise_stats['filtered_noise']} 개")
print(f"  [전처리 통계] 최종 저장된 텍스트 청크: {len(pdf_text_chunks)} 개")
print(f"  ------------------------------------------")
print(f"  └─ 수치 요약 청크(summary) : {len([c for c in pdf_text_chunks if c['chunk_type'] == 'summary'])} 개")
print(f"  └─ 일반 문맥 청크(normal)  : {len([c for c in pdf_text_chunks if c['chunk_type'] == 'normal'])} 개")
print(f"  ==========================================")

텍스트 청킹 중: 100%|██████████| 25/25 [00:00<00:00, 151.32it/s]


  텍스트 청킹 및 정제 완료
  [전처리 통계] 최초 분할된 총 청크 수 : 3415 개
  [전처리 통계] 길이 미달로 제거된 청크: 34 개
  [전처리 통계] 노이즈 패턴으로 제거됨  : 178 개
  [전처리 통계] 최종 저장된 텍스트 청크: 3203 개
  ------------------------------------------
  └─ 수치 요약 청크(summary) : 1317 개
  └─ 일반 문맥 청크(normal)  : 1886 개


In [55]:
print("=== 전체 청크 품질 확인 ===\n")
print(f"  표 청크 수     : {len(all_table_chunks)}개")
print(f"  텍스트 청크 수 : {len(pdf_text_chunks)}개")
print(f"  전체 청크 수   : {len(all_table_chunks) + len(pdf_text_chunks)}개")
print()

# 표 샘플 확인
print("=== 표 청크 샘플 ===")
for chunk in all_table_chunks[:2]:
    print(f"[{chunk['metadata']['파일명']} — {chunk['metadata']['page']}페이지]")
    print(chunk["text"][:200])
    if chunk["numeric_summary"]:
        print(f"numeric_summary: {chunk['numeric_summary']}")
    print()

# 텍스트 청크 샘플 확인
print("=== 텍스트 청크 샘플 ===")
for chunk in pdf_text_chunks[:2]:
    print(f"chunk_id  : {chunk['chunk_id']}")
    print(f"chunk_type: {chunk['chunk_type']}")
    print(f"text      : {chunk['text'][:150]}")
    if chunk["numeric_summary"]:
        print(f"summary   : {chunk['numeric_summary']}")
    print()

=== 전체 청크 품질 확인 ===

  표 청크 수     : 3664개
  텍스트 청크 수 : 3203개
  전체 청크 수   : 6867개

=== 표 청크 샘플 ===
[고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf — 5페이지]
| 정보시스템 | 주요내용 | 비고 |
| --- | --- | --- |
| 포털시스템 | ­ 통합로그인, 통합/지능형 검색, 마이페이지, 공지/알림, 일정관리,
커뮤니티, 게시판, 사용자별 정보서비스, 위젯, 연계서비스(웹메일,
챗봇, 전자결재, 학사/행정 서비스) 등
­ 학생(졸업생포함), 교직원, 연구원 등 내부 구성원 대상 포털
­ 학생/교수 등 

[고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf — 6페이지]
| 전자계약시스템 | - 인사, 구매, 공사, 학사 등 학내 계약건 전체 적용
­ 전자계약 관련 계약서 주서식/부서식/관련서류 관리
­ 계약 유형별 사전 정의된 계약 서식에 계약 정보 작성
­ 계약/서명 정보 전송 및 진행상태 관리
­ 계약업무 진행 시 각 단계마다 포탈 전자결재와 연동하여 결재
진행상태 관리
­ 계약체결이 완료된 계약서 조회 및 출력 기능

=== 텍스트 청크 샘플 ===
chunk_id  : pdf_text_10000_0
chunk_type: summary
text      : 제 안 요 청 서
고려대학교
차세대 포털·학사 정보시스템 구축 사업
2024. 7. 01
※ 본 자료는 제안내용의 설명을 위한 배포자료로, 이외 목적으로 무단복제, 전달 및 사용하는 행위를 금함.
-1-
summary   : 날짜: 2024. 7.

chunk_id  : pdf_text_10000_1
chunk_type: summary
text      : I 사업 개요
1. 사업개요
가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업
나. 사업기간: 계약일로부터 24개월 이내
다. 무상유지보수기간 : 사업종료일로부터 12개월
라. 사업예산 : 11,270,000,000원 (V.A.T 포

In [56]:
total_tables = len(all_table_chunks)
total_texts = len(pdf_text_chunks)
total_chunks = total_tables + total_texts

table_summary_count = len([c for c in all_table_chunks if c["chunk_type"] == "summary"])
text_summary_count = len([c for c in pdf_text_chunks if c["chunk_type"] == "summary"])
total_numeric_chunks = table_summary_count + text_summary_count

print("==================================================")
print("            RFP 파이프라인 최종 요약 지표          ")
print("==================================================")
print(f"1. 총 처리된 PDF 파일 수 : {len(pdf_df)} 개")
print(f"2. 생성된 총 Chunk 수    : {total_chunks} 개")
print(f"   - 표(Table) 청크     : {total_tables} 개 (구조화 완료)")
print(f"   - 텍스트 청크        : {total_texts} 개")
print(f"3. 수치 데이터 매칭 정보 : 총 {total_numeric_chunks} 개 청크에서 핵심 수치 추출")
print(f"   - 표 내 수치 추출    : {table_summary_count} 개")
print(f"   - 본문 내 수치 추출  : {text_summary_count} 개")
print(f"4. 데이터 압축 및 정제율 : {((noise_stats['filtered_short'] + noise_stats['filtered_noise']) / noise_stats['total_splits']) * 100:.2f}% 제거 완료")
print("==================================================")

            RFP 파이프라인 최종 요약 지표          
1. 총 처리된 PDF 파일 수 : 25 개
2. 생성된 총 Chunk 수    : 6867 개
   - 표(Table) 청크     : 3664 개 (구조화 완료)
   - 텍스트 청크        : 3203 개
3. 수치 데이터 매칭 정보 : 총 2022 개 청크에서 핵심 수치 추출
   - 표 내 수치 추출    : 705 개
   - 본문 내 수치 추출  : 1317 개
4. 데이터 압축 및 정제율 : 6.21% 제거 완료


---
## 4. HWP 청킹


In [57]:
def is_toc_chunk(text: str) -> bool:
    """목차성 청크 판별 — True면 제거 대상"""
    
    # 사업 개요 키워드가 있으면 목차여도 제거하지 않음
    if any(kw in text for kw in ["사업목적", "사업기간", "사업내용", "추진목적",
                                  "과업개요", "과업명"]):
        return False
    
    if '목 차' in text or '목차' in text:
        return True
    if len(re.findall(r'(?<!\d)\d+\.\s+\S', text)) >= 4:
        return True
    if len(re.findall(r'\)\s*\d+', text)) >= 3:
        return True
    if len(re.findall(r'[ⅠⅡⅢⅣⅤ]', text)) >= 2 and len(re.findall(r'\s\d{1,3}\s', text)) >= 4:
        return True
    total = len(text.replace(' ', ''))
    if total > 0 and len(re.findall(r'\d', text)) / total > 0.20:
        return True
    return False

print('✅ 목차 필터 함수 정의 완료')


✅ 목차 필터 함수 정의 완료


In [58]:
CHUNK_SIZE    = 1024
CHUNK_OVERLAP = 200

In [59]:
# ── 표 블록 보호 청킹 ────────────────────────────────────────────
# Markdown 표 행(| 로 시작)이 청크 경계에서 잘리지 않도록 후처리

def split_preserving_tables(text: str, splitter) -> list:
    raw_chunks = splitter.split_text(text)
    result = []
    carry  = ""

    for chunk in raw_chunks:
        chunk = carry + chunk
        carry = ""
        lines = chunk.split("\n")

        # 청크 끝이 표 안에 걸쳐 있으면 표 블록 전체를 carry로 넘김
        last_table_line = -1
        for i in range(len(lines) - 1, -1, -1):
            if lines[i].startswith("|"):
                last_table_line = i
                break

        if last_table_line != -1:
            table_start = last_table_line
            while table_start > 0 and lines[table_start - 1].startswith("|"):
                table_start -= 1
            body  = "\n".join(lines[:table_start]).strip()
            table = "\n".join(lines[table_start:]).strip()
            if body:
                result.append(body)
            carry = table + "\n"
        else:
            result.append(chunk)

    if carry.strip():
        result.append(carry.strip())

    # 최대 길이 초과 청크 강제 분할
    final_result = []
    for chunk in result:
        if len(chunk) <= 2000:
            final_result.append(chunk)
        else:
            # 2000자 단위로 강제 분할
            for i in range(0, len(chunk), 2000):
                part = chunk[i:i+2000].strip()
                if part:
                    final_result.append(part)
    return final_result

def is_real_table(chunk: str) -> bool:
    """실제 데이터 표 여부 판단 — 2행 이상, 2열 이상"""
    table_lines = [l for l in chunk.split("\n") if l.startswith("|")]
    if len(table_lines) < 3:  # 헤더 + 구분선 + 데이터 최소 3행
        return False
    # 열 수 확인 — 구분선 행 기준
    separator_lines = [l for l in table_lines if "---" in l]
    if not separator_lines:
        return False
    cols = separator_lines[0].count("---")
    return cols >= 2

def chunk_document(doc: dict) -> list:
    doc_id = doc["doc_id"]
    chunks = split_preserving_tables(doc["text"], splitter)
    result, idx = [], 0

    for chunk in chunks:
        if is_toc_chunk(chunk):
            continue
        if len(chunk) < 50:
            continue

        found   = find_numbers_hwp(chunk)
        numeric = make_numeric_summary(found) if found else ""

        if is_real_table(chunk):
            chunk_type = "table"
        elif found:
            chunk_type = "summary"
        else:
            chunk_type = "normal"

        result.append({
            "chunk_id"       : f"{doc_id}_{idx}",
            "doc_id"         : doc_id,
            "chunk_index"    : idx,
            "text"           : chunk,
            "numeric_summary": numeric,
            "chunk_type"     : chunk_type,
            "metadata": {
                "doc_id"       : str(doc_id),
                "사업명"       : doc["사업명"],
                "발주기관"     : doc["발주기관"],
                "사업금액"     : doc["사업금액"],
                "공고번호"     : doc["공고번호"],
                "파일형식"     : doc["파일형식"],
                "has_full_text": doc["has_full_text"],
                "has_table"    : doc["has_table"],
                "chunk_id"     : f"{doc_id}_{idx}",
                "chunk_index"  : str(idx),
                "section": extract_section_name(chunk)  ############ 추가
            }
        })
        idx += 1
    return result


all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))

table_chunks   = [c for c in all_chunks if c["chunk_type"] == "table"]
summary_chunks = [c for c in all_chunks if c["chunk_type"] == "summary"]
normal_chunks  = [c for c in all_chunks if c["chunk_type"] == "normal"]

print(f"✅ 청킹 완료 (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
print(f"  총 문서 수  : {len(documents)}개")
print(f"  총 청크 수  : {len(all_chunks)}개")
print(f"  문서당 평균 : {len(all_chunks)/len(documents):.1f}개")
print()
print(f"  표 청크     : {len(table_chunks)}개  ({len(table_chunks)/len(all_chunks)*100:.1f}%)")
print(f"  수치 청크   : {len(summary_chunks)}개 ({len(summary_chunks)/len(all_chunks)*100:.1f}%)")
print(f"  일반 청크   : {len(normal_chunks)}개 ({len(normal_chunks)/len(all_chunks)*100:.1f}%)")

✅ 청킹 완료 (chunk_size=1024, overlap=200)
  총 문서 수  : 665개
  총 청크 수  : 3234개
  문서당 평균 : 4.9개

  표 청크     : 0개  (0.0%)
  수치 청크   : 1224개 (37.8%)
  일반 청크   : 2010개 (62.2%)


In [60]:
first_doc_chunks = [c for c in all_chunks if c["doc_id"] == 0]
print(f"첫 번째 문서 청크 수: {len(first_doc_chunks)}개")
print()
for i, chunk in enumerate(first_doc_chunks[:3]):
    print(f"=== 청크 {i+1} (chunk_id: {chunk['chunk_id']}) ===")
    print(f"글자 수: {len(chunk['text'])}자")
    print(f"chunk_type: {chunk['chunk_type']}")
    if chunk['numeric_summary']:
        print(f"numeric_summary: {chunk['numeric_summary']}")
    print(chunk["text"][:200])
    print()


첫 번째 문서 청크 수: 2개

=== 청크 1 (chunk_id: 0_0) ===
글자 수: 988자
chunk_type: summary
numeric_summary: 금액: 130,000,000원 | 기간: 1개월, 3개월 | 수량: 1개, 3개
[사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 | 발주기관: 한영대학]

2024년 특성화 맞춤형 교육환경 구축 트랙운영 학사정보시스템 고도화
 제안요청서

2024. 10.

목 차
Ⅰ. 사업 안내 - 1
 1. 사업개요 - 1
 2. 추진배경 및 필요성 - 1
 3. 기대효과 - 1
 Ⅱ. 구축 방안 - 2
 1. 

=== 청크 2 (chunk_id: 0_1) ===
글자 수: 484자
chunk_type: summary
numeric_summary: 기간: 1개월, 3개월 | 수량: 1개, 3개
3
 기대효과
 트랙기반 교육과정의 운영 및 관리 체계를 효과적으로 지원
 교수자-학습자 중심의 교육환경 조성을 통한 대학 교육의 가치 구현
 학사운영 시스템을 통해 대학 체제 개편에 대한 대응체계 확립

Ⅱ 
 구축 방안

1 
 구축목표
 트랙제도 교과과정 개편 및 표준운영관리를 위한 시스템 구현
 (교과과정 개발에서 성적 이수까지)
 트랙제도 기반 교



In [61]:
import statistics as st

chunk_lengths = [len(c["text"]) for c in all_chunks]
summary_chunks = [c for c in all_chunks if c["chunk_type"] == "summary"]
normal_chunks  = [c for c in all_chunks if c["chunk_type"] == "normal"]

print("=== 청크 글자 수 통계 ===")
print(f"  최소  : {min(chunk_lengths):,}자")
print(f"  최대  : {max(chunk_lengths):,}자")
print(f"  평균  : {st.mean(chunk_lengths):,.0f}자")
print(f"  중앙값: {st.median(chunk_lengths):,.0f}자")
print()
print(f"수치 요약 청크 : {len(summary_chunks)}개 ({len(summary_chunks)/len(all_chunks)*100:.1f}%)")
print(f"일반 청크      : {len(normal_chunks)}개 ({len(normal_chunks)/len(all_chunks)*100:.1f}%)")


=== 청크 글자 수 통계 ===
  최소  : 61자
  최대  : 2,000자
  평균  : 739자
  중앙값: 817자

수치 요약 청크 : 1224개 (37.8%)
일반 청크      : 2010개 (62.2%)


---
## 5. 통합 및 저장


In [62]:
# PDF 텍스트 청크 스키마를 HWP 기준으로 통일
# all_table_chunks 는 추출 단계에서 이미 HWP 스키마로 저장됨
# pdf_text_chunks 스키마 확인 및 통일

unified_pdf_chunks = []

# 표 청크 — 이미 HWP 스키마, 그대로 사용
for chunk in all_table_chunks:
    unified_pdf_chunks.append(chunk)

# 텍스트 청크 — 스키마 통일
for chunk in pdf_text_chunks:
    unified_pdf_chunks.append({
        "chunk_id"       : chunk.get("chunk_id", ""),
        "doc_id"         : chunk.get("doc_id", ""),
        "chunk_index"    : chunk.get("chunk_index", 0),
        "text"           : chunk.get("text", ""),
        "numeric_summary": chunk.get("numeric_summary", ""),
        "chunk_type"     : chunk.get("chunk_type", "normal"),
        "metadata": {
            "doc_id"       : str(chunk.get("doc_id", "")),
            "사업명"       : chunk.get("metadata", {}).get("사업명", ""),
            "발주기관"     : chunk.get("metadata", {}).get("발주기관", ""),
            "사업금액"     : chunk.get("metadata", {}).get("사업금액", ""),
            "공고번호"     : chunk.get("metadata", {}).get("공고번호", ""),
            "파일형식"     : "pdf",
            "파일명"       : chunk.get("metadata", {}).get("파일명", ""),
            "has_full_text": True,
            "has_table"    : False,
            "chunk_id"     : chunk.get("chunk_id", ""),
            "chunk_index"  : str(chunk.get("chunk_index", 0)),
            "section"     : chunk.get("section", ""),  ################ 추가
            "page"    : chunk.get("metadata", {}).get("page", ""),  # 추가
        }
    })

print(f"✅ PDF 스키마 통일 완료")
print(f"  표 청크     : {len(all_table_chunks)}개")
print(f"  텍스트 청크 : {len(pdf_text_chunks)}개")
print(f"  통합 PDF 청크: {len(unified_pdf_chunks)}개")

# 스키마 샘플 확인
print()
print("=== 스키마 확인 (표 청크 샘플) ===")
print(list(unified_pdf_chunks[0].keys()))
print()
print("=== 스키마 확인 (텍스트 청크 샘플) ===")
print(list(unified_pdf_chunks[-1].keys()))

✅ PDF 스키마 통일 완료
  표 청크     : 3664개
  텍스트 청크 : 3203개
  통합 PDF 청크: 6867개

=== 스키마 확인 (표 청크 샘플) ===
['chunk_id', 'doc_id', 'chunk_index', 'text', 'numeric_summary', 'chunk_type', 'metadata']

=== 스키마 확인 (텍스트 청크 샘플) ===
['chunk_id', 'doc_id', 'chunk_index', 'text', 'numeric_summary', 'chunk_type', 'metadata']


In [63]:
# HWP 청크 + PDF 청크 통합
all_combined_chunks = all_chunks + unified_pdf_chunks

print(f"✅ 통합 완료")
print(f"  HWP 청크  : {len(all_chunks)}개")
print(f"  PDF 청크  : {len(unified_pdf_chunks)}개")
print(f"  총 청크   : {len(all_combined_chunks)}개")
print()

# 타입별 분포
table_total   = sum(1 for c in all_combined_chunks if c["chunk_type"] == "table")
summary_total = sum(1 for c in all_combined_chunks if c["chunk_type"] == "summary")
normal_total  = sum(1 for c in all_combined_chunks if c["chunk_type"] == "normal")
print(f"  표 청크    : {table_total}개  ({table_total/len(all_combined_chunks)*100:.1f}%)")
print(f"  수치 청크  : {summary_total}개 ({summary_total/len(all_combined_chunks)*100:.1f}%)")
print(f"  일반 청크  : {normal_total}개 ({normal_total/len(all_combined_chunks)*100:.1f}%)")

✅ 통합 완료
  HWP 청크  : 3234개
  PDF 청크  : 6867개
  총 청크   : 10101개

  표 청크    : 2959개  (29.3%)
  수치 청크  : 3246개 (32.1%)
  일반 청크  : 3896개 (38.6%)


In [64]:
import json
OUTPUT_DIR = os.path.join(BASE_DIR, "data")

# HWP + PDF 통합 청크 저장
with open(os.path.join(OUTPUT_DIR, "chunks_data.json"), "w", encoding="utf-8") as f:
    json.dump(all_combined_chunks, f, ensure_ascii=False, indent=2)

# HWP 문서 구조화 데이터 저장
with open(os.path.join(OUTPUT_DIR, "extracted_data.json"), "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

hwp_count = sum(1 for c in all_combined_chunks if c['metadata'].get('파일형식') == 'hwp')
pdf_count = sum(1 for c in all_combined_chunks if c['metadata'].get('파일형식') == 'pdf')

print(f"✅ chunks_data.json 저장       : {len(all_combined_chunks):,}개 (HWP + PDF)")
print(f"   HWP 청크 : {hwp_count:,}개")
print(f"   PDF 청크 : {pdf_count:,}개")
print(f"✅ extracted_data.json 저장    : {len(documents)}건 (HWP)")
print(f"\n저장 위치: {OUTPUT_DIR}")

✅ chunks_data.json 저장       : 10,101개 (HWP + PDF)
   HWP 청크 : 3,234개
   PDF 청크 : 6,867개
✅ extracted_data.json 저장    : 665건 (HWP)

저장 위치: /Users/who/Desktop/code_it/project01/final_files/data


---
## 📋 데이터 처리 인수인계

### 📁 생성된 파일
| 파일명 | 내용 | 건수 |
|---|---|---|
| `chunks_data.json` | HWP + PDF 통합 청크 | HWP 44,598 + PDF 6,867 = **51,465개** |
| `extracted_data.json` | HWP 구조화 문서 | **665건** |

저장 위치: `~/AI-based-RFP-RAG-System/data/`

> **extracted_data.json 활용 안내**
> HWP 문서 구조화 데이터로, 청크로 쪼개기 전 원본 문서 정보가 담겨있습니다.
> - 특정 문서의 전체 내용을 한 번에 가져와야 할 때
> - 청크에서 원본 문서로 역추적이 필요할 때
> - 사업명, 발주기관, 사업금액 같은 문서 메타데이터를 별도로 참조할 때
>
> ※ PDF는 pdfplumber로 페이지 단위 직접 파싱하여 문서 구조화 단계 없이
> 표 청크와 텍스트 청크로 바로 분리되므로 별도 extracted_data 없음

---

### 🗂️ 청크 구조 (HWP/PDF 공통)
```python
"metadata": {
    # 공통
    "doc_id", "사업명", "발주기관", "사업금액", "공고번호",
    "파일형식", "파일명", "has_full_text", "has_table",
    "chunk_id", "chunk_index",
    # HWP만 해당
    "section",  # 섹션명
    # PDF만 해당
    "page",     # 페이지 번호
}
```

### ⚠️ PDF 섹션명 미지원
PDF 청크에는 `section` 필드가 없고 `page` 필드만 존재.

**원인**
PDF는 pdfplumber로 페이지 단위 파싱하여 섹션 감지 로직 없이
표/텍스트 청크로 바로 분리되는 구조라 섹션명 추출 불가.

**출처 표시 권장 방식**
- HWP: `사업명 | 발주기관 | 섹션명`
- PDF: `사업명 | 발주기관 | p.{page}`

**향후 개선 방향**: PDF 재처리 시 `Ⅰ.`, `1.`, `가.` 등 패턴 기반 섹션 감지 로직 추가 가능.
단, PDF마다 형식이 달라 HWP 대비 정확도 낮을 수 있음.

---

### ⚠️ PDF 목차 청크 미필터링
HWP는 청킹 후 `is_toc_chunk()`로 목차성 청크를 제거하지만, PDF는 적용하지 않음.

**현황**
- 목차 관련 PDF 청크 16개 확인 (전체 6,867개 중 0.2%)
- 대부분 표 안에 "목차" 단어가 포함된 경우로, 실제 목차 페이지는 소수

**미처리 사유**
- 영향이 미미하여 별도 처리 불필요 판단
- PDF 목차는 표 형태로 파싱되어 `chunk_type: table`로 분류됨

**향후 개선 방향**
필요 시 PDF 청크에도 `is_toc_chunk()` 적용 가능.
단, PDF 목차 패턴이 HWP와 다를 수 있어 별도 튜닝 필요.

---

### ⚙️ 청킹 설정
- HWP: chunk_size=1024, overlap=200
- PDF: chunk_size=1024, overlap=200

> chunk_size/overlap 최적값은 임베딩/검색 실험 결과 보고 팀 합의로 최종 결정 필요

---

### 📌 chunk_type 정의
| chunk_type | 설명 |
|---|---|
| `table` | 2열 이상, 3행 이상 실제 데이터 표 |
| `summary` | 수치 패턴(금액, 기간, 수량 등) 포함 청크 |
| `normal` | 일반 텍스트 청크 |

In [65]:
# Cell 30 — 패키지 설치
import sys
!{sys.executable} -m pip install chromadb sentence-transformers einops -q
print("✅ chromadb 설치 완료")

✅ chromadb 설치 완료


In [66]:
# Cell 31 — 임베딩 모델 로드 + ChromaDB 초기화
import re
import torch
import chromadb
from sentence_transformers import SentenceTransformer

CHROMA_PATH     = os.path.join(BASE_DIR, "chroma_seol_qwen3")
COLLECTION_NAME = "rfp_seol_chunks_qwen3"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
EMBED_BATCH_SIZE = 8

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"🔄 임베딩 모델 로드 중... (device={DEVICE})")
emb_model = SentenceTransformer(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    device=DEVICE
)
emb_model.max_seq_length = 1024

def get_embeddings(texts):
    prepared = [re.sub(r"\s+", " ", str(t)).strip()[:3000] for t in texts]
    return emb_model.encode(
        prepared,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ Cell 31 완료 — 현재 저장된 청크: {collection.count():,}개")

🔄 임베딩 모델 로드 중... (device=mps)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4176.61it/s]


✅ Cell 31 완료 — 현재 저장된 청크: 900개


In [67]:
# Cell 32 — ChromaDB 구축 (재구축 포함)

import time

# 기존 컬렉션 삭제 후 재구축
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"🗑️  기존 컬렉션 삭제: {COLLECTION_NAME}")

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# 50자 미만 필터링
chunks_to_add = [c for c in all_combined_chunks if len(c.get("text", "").strip()) >= 50]
print(f"필터링: {len(all_combined_chunks):,}개 → {len(chunks_to_add):,}개 ({len(all_combined_chunks)-len(chunks_to_add):,}개 제거)")

print(f"\n⏳ 구축 시작: {len(chunks_to_add):,}개 청크")
t0 = time.time()

for i in range(0, len(chunks_to_add), 100):
    batch = chunks_to_add[i:i+100]
    texts = [c["text"] for c in batch]
    ids   = [c["chunk_id"] for c in batch]
    metadatas = []
    for c in batch:
        meta = {}
        for k, v in c["metadata"].items():
            meta[k] = "" if v is None else str(v)
        meta["chunk_type"] = c.get("chunk_type", "normal")
        metadatas.append(meta)

    collection.add(
        documents=texts,
        embeddings=get_embeddings(texts),
        metadatas=metadatas,
        ids=ids,
    )
    if (i // 100) % 20 == 0:
        elapsed = int(time.time() - t0)
        print(f"  [{i+len(batch):,}/{len(chunks_to_add):,}] {elapsed}s")

print(f"\n✅ Cell 32 완료 — {collection.count():,}개 저장됨")
print(f"   저장 경로: {CHROMA_PATH}")


🗑️  기존 컬렉션 삭제: rfp_seol_chunks_qwen3
필터링: 10,101개 → 10,002개 (99개 제거)

⏳ 구축 시작: 10,002개 청크
  [100/10,002] 33s
  [2,100/10,002] 758s
  [4,100/10,002] 1347s
  [6,100/10,002] 1761s
  [8,100/10,002] 2307s
  [10,002/10,002] 2947s

✅ Cell 32 완료 — 10,002개 저장됨
   저장 경로: /Users/who/Desktop/code_it/project01/final_files/chroma_seol_qwen3


In [68]:
# Cell 33 — ChromaDB 구축 검증

# 1. 기본 현황
print("=" * 50)
print("📊 ChromaDB 현황")
print("=" * 50)
print(f"  총 청크 수 : {collection.count():,}개")

# 2. 샘플 청크 확인
sample = collection.get(limit=5, include=["metadatas", "documents"])
print("\n📋 샘플 청크 (5개)")
for i, (doc, meta) in enumerate(zip(sample["documents"], sample["metadatas"])):
    print(f"  [{i+1}] {meta.get('발주기관', '')} / {meta.get('사업명', '')[:25]}")
    print(f"       chunk_type: {meta.get('chunk_type', '')} / {len(doc)}자")

# 3. 검색 테스트
print("\n" + "=" * 50)
print("🔍 검색 테스트")
print("=" * 50)
test_queries = [
    "학사정보시스템 고도화 예산",
    "입찰 참가 자격 요건",
    "사업기간 계약일로부터",
]
for query in test_queries:
    q_emb = get_embeddings([query])[0]
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=2,
        include=["documents", "metadatas", "distances"]
    )
    print(f"\n🔎 '{query}'")
    for j, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"  [{j+1}] {meta.get('발주기관','')} | {meta.get('사업명','')[:25]}")
        print(f"       유사도: {1-dist:.4f} / chunk_type: {meta.get('chunk_type','')}")
        print(f"       내용: {doc[:80]}...")

print("\n✅ Cell 33 완료 — 검증 완료")

📊 ChromaDB 현황
  총 청크 수 : 10,002개

📋 샘플 청크 (5개)
  [1] 한영대학 / 한영대학교 특성화 맞춤형 교육환경 구축 - 트
       chunk_type: summary / 988자
  [2] 한영대학 / 한영대학교 특성화 맞춤형 교육환경 구축 - 트
       chunk_type: summary / 484자
  [3] 한국연구재단 / 2024년 대학산학협력활동 실태조사 시스템(U
       chunk_type: summary / 800자
  [4] 한국연구재단 / 2024년 대학산학협력활동 실태조사 시스템(U
       chunk_type: summary / 699자
  [5] 한국연구재단 / 2024년 대학산학협력활동 실태조사 시스템(U
       chunk_type: normal / 1018자

🔍 검색 테스트

🔎 '학사정보시스템 고도화 예산'
  [1] 전라남도교육청 | 통합정보 관리시스템 고도화사업
       유사도: 0.6526 / chunk_type: summary
       내용: [사업명: 통합정보 관리시스템 고도화사업 | 발주기관: 전라남도교육청]

통합정보관리시스템 고도화 사업
제안 요청서

2024. 6.

(예산과...
  [2] 국립순천대학교 | 국립순천대학교 통합학사시스템(향림통) 고도화 
       유사도: 0.6519 / chunk_type: normal
       내용: [사업명: 국립순천대학교 통합학사시스템(향림통) 고도화 수의시담 | 발주기관: 국립순천대학교]

국립순천대학교 통합학사시스템(향림통) 고도화 사...

🔎 '입찰 참가 자격 요건'
  [1] 경기도 성남시 | 성남시 통합성과관리시스템(BSC) 재구축 용역
       유사도: 0.6761 / chunk_type: summary
       내용: Ⅲ
제안 안내
 1. 적용 규정
지방자치단체를 당사자로 하는 계약에 관한 법률 시행령(대통령령 제34494호) 제43조(협상에 의한 계약체결)
...
  [2] 성남시청 